# 3.10 · 划分与交叉验证 / Splitting & Cross-Validation

> **课程定位 / Where this fits**
> **Part 3 第 10 课**。3.9 反复说"按时间/分组划分"; 这一课系统讲清**怎么划分数据来诚实地估计泛化能力**。核心问题：**你的离线分数能不能信？** 划分方式直接决定答案。这也是 Part 5 模型评估的地基。
> How to split data to honestly estimate generalization. The split method determines whether your offline score is trustworthy.

> 💡 **面试相关 / Interview-relevant**
> - "为什么要交叉验证而不只是一次 train/test" ★★★★★
> - "K-fold 的 K 怎么选" ★★★★
> - "分层抽样什么时候必须用" ★★★★（不平衡分类）
> - "时序数据怎么做 CV" ★★★★★（TimeSeriesSplit）
> - "什么是 nested CV" ★★★（调参 + 评估分离）

---

## 学习目标 / Learning Objectives
1. 理解 **train/val/test 三分** 的角色分工（为什么需要三个）。
2. 掌握 **K-fold** 及其变体：分层 / 分组 / 时序。
3. 给每种数据特点**选对 CV 策略**（这是 3.9 防泄漏的执行层）。
4. 理解 **nested CV** 为什么是"调参 + 无偏评估"的金标准。

## 目录 / TOC
1. [为什么不能只 train/test 一次 ⭐](#1)
2. [🌸 数据](#2)
3. [K-fold 交叉验证](#3)
4. [分层 K-fold：不平衡必用 ⭐](#4)
5. [GroupKFold：防分组泄漏](#5)
6. [TimeSeriesSplit：时序专用 ⭐](#6)
7. [train/val/test 三分的角色](#7)
8. [Nested CV：调参的正确姿势 ⭐](#8)
9. [选择决策表](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 为什么不能只 train/test 一次 ⭐ / Why Not a Single Split

单次 train/test 的问题：**分数取决于"哪些样本碰巧进了 test"** —— 运气成分大, 尤其小数据。

**交叉验证 (K-fold)**：把数据切 $K$ 份, 轮流用 $K-1$ 份训练、1 份验证, **每个样本都当过一次验证**。最终分数 = $K$ 次的平均 ± 标准差。

| 单次划分 | K-fold CV |
|---|---|
| 1 个分数（运气大）| $K$ 个分数取平均（稳健）|
| 浪费数据（test 不参与训练）| 每个样本都训练+验证过 |
| 无法估计方差 | **给出 ±std**（分数的可信度）|


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     StratifiedKFold, GroupKFold, TimeSeriesSplit)
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

X, y = load_iris(return_X_y=True)

# 演示单次划分的"运气" / single-split luck
scores = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed)
    clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
    scores.append(clf.score(Xte, yte))
print(f"20 个不同随机种子的单次 test 准确率:")
print(f"  范围 [{min(scores):.1%}, {max(scores):.1%}], std={np.std(scores):.1%}")
print(f"  → 同一模型同一数据, 仅换 test 划分, 分数波动 {(max(scores)-min(scores))*100:.0f} 个百分点!")
print(f"\n5-fold CV (一次给出均值±std): ", end="")
cv = cross_val_score(LogisticRegression(max_iter=500), X, y, cv=5)
print(f"{cv.mean():.1%} ± {cv.std():.1%}")


**铁证**：同一模型、同一数据, **仅换 test 划分的随机种子**, 单次分数能波动好几个百分点——这就是为什么**报告单次分数不可信**。CV 用平均 ± std 给出诚实估计。
Same model, same data, just different test split — single-split scores swing several points. That's why CV's mean±std is honest.


<a id="3"></a>
## 3. K-fold 交叉验证 / K-fold CV

```
5-fold 示意 (■=验证, □=训练):
  fold 1: ■ □ □ □ □
  fold 2: □ ■ □ □ □
  fold 3: □ □ ■ □ □
  fold 4: □ □ □ ■ □
  fold 5: □ □ □ □ ■
  每个样本恰好当 1 次验证
```

**K 怎么选**：
| K | 偏差 | 方差 | 计算量 |
|---|---|---|---|
| 小 (3) | 高（训练数据少）| 低 | 快 |
| **5 / 10** ⭐ | 平衡 | 平衡 | 适中（业界默认）|
| n (留一 LOOCV) | 最低 | 高 + 极慢 | n 次训练 |

**默认 5 或 10**。数据极少时用 LOOCV, 数据大时用 3-5（省算力）。


In [ ]:
# 可视化 K-fold 划分 / visualize the fold structure
def plot_cv(cv, X, y, groups=None, ax=None, title=""):
    n = len(X)
    for i, (tr, te) in enumerate(cv.split(X, y, groups)):
        idx = np.zeros(n)
        idx[te] = 1   # 1 = test/val
        ax.scatter(range(n), [i]*n, c=idx, cmap="coolwarm", marker="_", lw=8, vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel("sample index"); ax.set_ylabel("fold")
    ax.set_yticks(range(cv.get_n_splits(X, y, groups)))

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
plot_cv(KFold(5), X, y, ax=axes[0], title="KFold (顺序切, 红=验证)")
plot_cv(KFold(5, shuffle=True, random_state=0), X, y, ax=axes[1], title="KFold(shuffle=True)")
plt.tight_layout(); plt.show()
print("⚠ Iris 按类别排序! 不 shuffle 的 KFold 会让某些折全是一个类 → 必须 shuffle 或用分层")


<a id="4"></a>
## 4. 分层 K-fold：不平衡必用 ⭐ / Stratified K-fold

**普通 KFold 的陷阱**：类别不平衡 / 数据按类排序时, 某些折可能**类别比例严重失衡甚至缺类**。

**StratifiedKFold**：保证**每折的类别比例 ≈ 整体比例**（3.10 = 2.4 分层抽样在 CV 里的应用！）。**分类任务的默认**——sklearn 的 `cross_val_score` 对分类器自动用它。
StratifiedKFold keeps each fold's class ratio ≈ the overall ratio — the default for classification (it's 2.4's stratified sampling applied to CV).


In [ ]:
# 制造不平衡数据看差别 / imbalanced data
y_imb = np.array([0]*180 + [1]*20)        # 90% / 10%
X_imb = rng.normal(size=(200, 4))

print("普通 KFold (无 shuffle) 各折的少数类(1)数量:")
for i, (tr, te) in enumerate(KFold(5).split(X_imb)):
    print(f"  fold {i}: test 里类1 = {(y_imb[te]==1).sum()} 个", end="")
print("\n  ← 某些折可能 0 个少数类! 无法评估\n")

print("StratifiedKFold 各折的少数类(1)数量:")
for i, (tr, te) in enumerate(StratifiedKFold(5).split(X_imb, y_imb)):
    print(f"  fold {i}: test 里类1 = {(y_imb[te]==1).sum()} 个", end="")
print("\n  ← 每折都有 4 个 (= 20/5), 比例保持 ✓")


<a id="5"></a>
## 5. GroupKFold：防分组泄漏 / GroupKFold

3.9 节见过。**同一实体（用户/病人/设备）的所有样本必须在同一折**——避免模型靠"认出实体"作弊。


In [ ]:
# 同病人多次就诊 (3.9 的场景) / repeated entities
groups = np.repeat(np.arange(40), 5)      # 40 个病人, 每人 5 条
X_g = rng.normal(size=(200, 4)); y_g = rng.integers(0, 2, 200)

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
plot_cv(KFold(5), X_g, y_g, ax=axes[0], title="KFold: 同病人散在多折 (泄漏!)")
plot_cv(GroupKFold(5), X_g, y_g, groups=groups, ax=axes[1], title="GroupKFold: 同病人锁定一折 ✓")
plt.tight_layout(); plt.show()

# 验证: GroupKFold 保证 train/test 无共享 group / no shared groups
for tr, te in GroupKFold(5).split(X_g, y_g, groups):
    assert len(set(groups[tr]) & set(groups[te])) == 0
print("GroupKFold 保证: 每折 test 的病人在 train 里完全不出现 ✓")


<a id="6"></a>
## 6. TimeSeriesSplit：时序专用 ⭐ / TimeSeriesSplit

3.9 的时间泄漏在 CV 层的解法。**绝不能用未来预测过去** → 训练集永远在验证集**之前**。

```
TimeSeriesSplit (扩展窗口):
  fold 1: [train    ][val]
  fold 2: [train       ][val]
  fold 3: [train          ][val]
  训练窗口逐步扩大, val 永远在 train 之后 (时间向右)
```


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3))
X_ts = np.arange(100).reshape(-1, 1); y_ts = np.arange(100)
plot_cv(KFold(5), X_ts, y_ts, ax=axes[0], title="❌ KFold: val 散布, 含'未来'训练'过去'")
plot_cv(TimeSeriesSplit(5), X_ts, y_ts, ax=axes[1], title="✅ TimeSeriesSplit: train 永在 val 之前")
plt.tight_layout(); plt.show()
print("TimeSeriesSplit 模拟真实部署: 用历史训练, 预测未来; 训练集随时间增长 (Part 14 时序详述)")


<a id="7"></a>
## 7. train/val/test 三分的角色 / The Three-way Split

**为什么需要三个集合**（很多人只分两个）：

| 集合 | 用途 | 关键 |
|---|---|---|
| **train** | 训练模型参数 | 模型从这里学 |
| **validation** | 调超参 / 选模型 | **被你反复看 → 会"污染"** |
| **test** | **最终一次性**验收 | 全程不碰, 只在最后看一次 |

**核心洞察**：你**用 val 调参的过程, 本身就在 val 上"过拟合"**（试了 50 组超参, 选了在 val 上最好的——这个"最好"有运气成分）。所以需要一个**从未参与任何决策**的 test 集做诚实的最终估计。
The act of tuning on validation overfits to validation — you need an untouched test set for an honest final estimate.

**CV 的位置**：CV 通常替代 train/val（在 train 集内部做 K-fold 调参）, **test 集依然单独留出**。


In [ ]:
# 演示"调参在 val 上过拟合" / tuning overfits validation
from sklearn.svm import SVC
X3, y3 = load_iris(return_X_y=True)
X_trainval, X_test, y_trainval, y_test = train_test_split(X3, y3, test_size=0.2, random_state=0)
X_tr, X_val, y_tr, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=0)

# 试很多超参, 选 val 上最好的 / try many, pick best on val
best_val, best_C = 0, None
for C in np.logspace(-2, 3, 40):
    acc = SVC(C=C).fit(X_tr, y_tr).score(X_val, y_val)
    if acc > best_val: best_val, best_C = acc, C

test_acc = SVC(C=best_C).fit(X_trainval, y_trainval).score(X_test, y_test)
print(f"最优 C={best_C:.2f}")
print(f"它在 val 上的准确率: {best_val:.1%}  ← 乐观 (我们专门选的它)")
print(f"它在 test 上的准确率: {test_acc:.1%}  ← 这才是诚实估计")
print("\nval 分数偏乐观因为 C 是'挑'出来的; test 全程没参与挑选 → 无偏")


<a id="8"></a>
## 8. Nested CV：调参的正确姿势 ⭐ / Nested CV

**问题**：用 CV 调参, 再用**同一个 CV** 报告分数 → 调参偷看了所有折 → 分数虚高（3.9 预处理泄漏的近亲）。

**Nested CV（嵌套交叉验证）**：
- **外层 CV**：估计泛化能力（每个外层 fold 的 test 从不参与调参）
- **内层 CV**：在外层的 train 上调超参

```
外层 fold (报告分数)
  └─ 内层 fold (调超参) → 选最优 → 在外层 test 上评估
```

**金标准, 但贵**（K_outer × K_inner × n_params 次训练）。小数据/论文用; 大数据用单层 CV 调参 + 独立 test。


In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.datasets import make_classification

# 用有噪声的小数据 (Iris 太干净, 显不出乐观偏差) / noisy small data
X4, y4 = make_classification(n_samples=200, n_features=20, n_informative=5,
                             n_redundant=2, flip_y=0.15, random_state=0)
param_grid = {"C": np.logspace(-3, 3, 12)}

# 跨多个种子平均, 让乐观偏差稳定显现 / average over seeds
non_nested_scores, nested_scores = [], []
for seed in range(8):
    inner = StratifiedKFold(5, shuffle=True, random_state=seed)
    outer = StratifiedKFold(5, shuffle=True, random_state=seed)
    grid = GridSearchCV(SVC(), param_grid, cv=inner)
    grid.fit(X4, y4)
    non_nested_scores.append(grid.best_score_)              # 调参+报分同一CV
    nested_scores.append(cross_val_score(grid, X4, y4, cv=outer).mean())  # 嵌套
non_nested, nested = np.mean(non_nested_scores), np.mean(nested_scores)

print(f"非嵌套 CV (调参+报分同一CV): {non_nested:.1%}  ← 乐观偏差")
print(f"嵌套 CV (内层调参, 外层评估): {nested:.1%}  ← 无偏估计")
print(f"\n差异 {(non_nested-nested)*100:.1f} 个百分点 = 调参偷看带来的乐观偏差")
print("→ 要发表/汇报模型真实能力时用 nested CV; 选完超参再用全部 train 重训上线")


<a id="9"></a>
## 9. 选择决策表 / Decision Table

```
分类任务?              → StratifiedKFold (cross_val_score 对分类器自动用) ⭐
有重复实体(用户/病人)?  → GroupKFold (3.9 防分组泄漏)
时序数据?              → TimeSeriesSplit (训练永在验证前) ⭐
回归 + 独立同分布?      → KFold (shuffle=True)
分类 + 重复实体?        → StratifiedGroupKFold (两者兼顾)
要无偏报告模型能力?     → Nested CV (外评估内调参)
数据极少 (<100)?       → LOOCV (留一)

K 的默认: 5 或 10
永远: 留一个全程不碰的 test 做最终验收
```


<a id="10"></a>
## 10. 小结 / Summary

```
单次划分不可信 (运气大) → K-fold CV 给均值±std
K-fold 变体 (= 3.9 防泄漏的执行层):
  StratifiedKFold — 分类默认, 保类别比例 ⭐
  GroupKFold      — 重复实体, 防分组泄漏
  TimeSeriesSplit — 时序, 训练永在验证前 ⭐
三分 train/val/test: val 调参会被污染 → test 留作一次性诚实验收
Nested CV: 内层调参 + 外层评估 = 无偏的金标准 (贵)
K 默认 5/10; 永远留独立 test
```

### 💡 面试速查
1. **为什么 CV**: 单次划分运气大, CV 给均值±std + 用满数据
2. **分类必用 StratifiedKFold**: 保每折类别比例
3. **时序用 TimeSeriesSplit**: 永不用未来预测过去
4. **重复实体用 GroupKFold**: 防认出实体作弊
5. **Nested CV**: 调参偷看会乐观偏差, 嵌套才无偏

### 下一节
**3.11 不平衡数据**——4 节的分层 CV 处理了"评估时的不平衡", 但**训练时**的极端不平衡（欺诈 0.1%）还需专门武器：SMOTE / 类权重 / 阈值调整。
